# 02 · Indexing: turning 10,943 chunks into searchable vectors and keywords
Inspects the outputs of `ragtraining.index` (Phase 2). Two **dense** indexes and one **sparse** index:
- **Qwen3-Embedding 0.6B** (1024-d, normalized) → main dense retriever (**D13**)
- **nomic-embed-text raw** (768-d, NOT normalized) → only for the cosine-vs-Euclid experiment (**D1/D14**)
- **BM25** with Snowball stemming → keyword retriever (**D15**)

In [ ]:
import sys, json, pickle; sys.path.insert(0, '..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.max_colwidth', 70); pd.set_option('display.width', 200)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
chunks = pd.read_parquet('../data/chunks.parquet')
D = np.load('../data/emb_qwen3.npy'); R = np.load('../data/emb_nomic_raw.npy')
meta = json.load(open('../data/index_meta.json'))
print('chunks', len(chunks), '| Qwen3', D.shape, '| nomic raw', R.shape)

## 1. Vector length (norm): does it carry information?
Cosine only looks at **direction**, Euclidean distance also at **length**. So combining them can only help if lengths
vary in a meaningful way. Ollama's `/api/embed` always returns unit vectors, and only the legacy endpoint gives raw ones (P-23).

In [ ]:
nd, nr = np.linalg.norm(D, axis=1), np.linalg.norm(R, axis=1)
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
ax[0].hist(nd, bins=50, range=(0.99, 1.01), color='tab:blue'); ax[0].set_title(f'Qwen3 norms: all = 1 (CV {nd.std()/nd.mean():.4f})')
ax[1].hist(nr, bins=60, color='tab:orange'); ax[1].set_title(f'nomic raw norms (CV {nr.std()/nr.mean():.3f})')
ax[2].scatter(chunks.n_tokens, nr, s=2, alpha=.25, color='tab:orange')
ax[2].set(xlabel='chunk tokens', ylabel='nomic norm', title='length of text vs length of vector')
plt.tight_layout(); plt.show()
print('correlation(tokens, norm) =', round(float(np.corrcoef(chunks.n_tokens, nr)[0, 1]), 3))

**Reading:** nomic lengths vary by only ~4% (CV). Shorter chunks get slightly longer vectors, which is a property
of the model, not of meaning. That's the first hint that Euclid won't beat cosine (confirmed in notebook 04).

## 2. The cosine ↔ Euclid identity, checked numerically
For unit vectors: **‖a − b‖² = 2 − 2·cos θ**, so both give the *same ranking*. On raw vectors they can differ.

In [ ]:
rng = np.random.default_rng(0); i, j = rng.integers(0, len(D), (2, 2000))
cos_q = (D[i] * D[j]).sum(1); d2_q = ((D[i] - D[j]) ** 2).sum(1)
print('Qwen3 (unit):  max |d² - (2 - 2cos)| =', f'{np.abs(d2_q - (2 - 2 * cos_q)).max():.2e}')
cos_n = (R[i] * R[j]).sum(1) / (nr[i] * nr[j]); dist_n = np.linalg.norm(R[i] - R[j], axis=1)
rho = pd.Series(cos_n).corr(pd.Series(-dist_n), method='spearman')
print(f'nomic (raw):   rank correlation(cos, -distance) = {rho:.3f}  (1.000 would mean identical rankings)')

## 3. A map of the corpus
Each dot is a chunk. **PCA** reduces 1024 → 50 dimensions, then **t-SNE** squeezes them to 2-D while keeping
neighbours close. Courses form their own islands. Overlaps show topics shared across courses
(e.g. CAPM appears in Investments, Financial Modeling and Financial Intermediation).

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
s = chunks.sample(3000, random_state=0)
xy = TSNE(2, init='pca', perplexity=30, random_state=0).fit_transform(PCA(50, random_state=0).fit_transform(D[s.index]))
fig, ax = plt.subplots(figsize=(11, 7.5))
cmap = plt.get_cmap('tab20')
for k, (course, idx) in enumerate(s.groupby('course').groups.items()):
    m = s.index.get_indexer(idx)
    ax.scatter(xy[m, 0], xy[m, 1], s=4, color=cmap(k % 20), label=f'{course[:28]} ({len(m)})')
ax.legend(markerscale=4, fontsize=7, bbox_to_anchor=(1.01, 1), loc='upper left'); ax.set_xticks([]); ax.set_yticks([])
ax.set_title('3,000 random chunks, Qwen3 embeddings, t-SNE'); plt.tight_layout(); plt.show()

## 4. The sparse side: what BM25 actually sees
Text is lower-cased, stop words dropped, words stemmed ("locking" → "lock"). **IDF** = how rare a term is in the corpus:
rare terms (2pl, serializ) decide the ranking, and common ones (use, data) barely count.

In [ ]:
from ragtraining.index.sparse import tokenize
bm = pickle.load(open('../data/bm25.pkl', 'rb'))
q = 'How does two-phase locking (2PL) guarantee conflict serializability of a schedule?'
print('raw tokens :', tokenize(q, stem=False)); print('stemmed    :', tokenize(q))
idf = bm.bm25.idf
pd.DataFrame({'term': tokenize(q), 'IDF': [round(idf.get(t, 0), 2) for t in tokenize(q)]}).sort_values('IDF', ascending=False)

In [ ]:
print(f"BM25 parameters: k1={meta['sparse']['k1']} (term-frequency saturation), b={meta['sparse']['b']} (length penalty)")
print('vocabulary size:', len(idf), 'stemmed terms')